In [ ]:
import cv2
import numpy as np
import pandas as pd
import yt_dlp
from moviepy.editor import VideoFileClip
import librosa
from youtube_transcript_api import YouTubeTranscriptApi
import re
from collections import Counter
import os

# -----------------------------
# 1. DOWNLOAD VIDEO
# -----------------------------
def download_video(url, filename_prefix):
    ydl_opts = {
        'outtmpl': f'{filename_prefix}.%(ext)s',
        'format': 'mp4'
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)

            video_id = info.get('id')
            title = info.get('title', 'unknown')

            return video_id, title, f"{filename_prefix}.mp4"

    except Exception as e:
        print(f"Download failed: {e}")
        return None, None, None


# -----------------------------
# 2. FRAME EXTRACTION
# -----------------------------
def extract_frames(video_path, interval=30):
    cap = cv2.VideoCapture(video_path)
    frames = []
    i = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if i % interval == 0:
            frames.append(frame)
        i += 1

    cap.release()
    return frames


# -----------------------------
# 3. VISUAL ANALYSIS
# -----------------------------
def average_brightness(frames):
    if not frames:
        return 0

    values = []
    for frame in frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        values.append(np.mean(gray))

    return float(np.mean(values))

def count_cuts(video_path):
    cap = cv2.VideoCapture(video_path)
    prev = None
    cuts = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev is not None:
            diff = cv2.absdiff(prev, gray)
            if np.mean(diff) > 30:
                cuts += 1

        prev = gray

    cap.release()
    return cuts

def average_colorfulness(frames):
    if not frames:
        return 0

    colorfulness_scores = []
    for frame in frames:
        # Convert BGR to LAB color space
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        # Extract a and b channels
        a = lab[:, :, 1]
        b = lab[:, :, 2]

        # Compute standard deviations
        std_a = np.std(a)
        std_b = np.std(b)

        # Compute means
        mean_a = np.mean(a)
        mean_b = np.mean(b)

        # Calculate colorfulness using Ruderman's formula
        colorfulness_score = np.sqrt((std_a ** 2) + (std_b ** 2)) + (0.3 * np.sqrt((mean_a ** 2) + (mean_b ** 2)))
        colorfulness_scores.append(colorfulness_score)

    return float(np.mean(colorfulness_scores)) if colorfulness_scores else 0

def motion_intensity(video_path, sample_rate=10):
    cap = cv2.VideoCapture(video_path)
    motion_values = []
    prev_gray = None
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_rate == 0: # Sample frames to reduce computation
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if prev_gray is not None:
                diff = cv2.absdiff(prev_gray, gray)
                motion_values.append(np.mean(diff))
            prev_gray = gray
        frame_idx += 1

    cap.release()
    return float(np.mean(motion_values)) if motion_values else 0


# -----------------------------
# 4. AUDIO ANALYSIS
# -----------------------------
def extract_audio(video_path, audio_path):
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path, verbose=False, logger=None)


def audio_energy(audio_path):
    y, sr = librosa.load(audio_path)
    return float(np.mean(librosa.feature.rms(y=y)))


# -----------------------------
# 5. TRANSCRIPT
# -----------------------------
def get_transcript(video_id):
    try:
        api = YouTubeTranscriptApi()
        transcript_list = api.list(video_id)

        try:
            # Try to find English (en) or English (United Kingdom) (en-GB)
            transcript = transcript_list.find_transcript(['en', 'en-GB'])
        except Exception:
            # Fallback to finding any available transcript if specific English not found
            available_languages = [t.language_code for t in transcript_list]
            if available_languages:
                transcript = transcript_list.find_transcript(available_languages)
            else:
                raise Exception("No transcripts found for any language.")

        return transcript.fetch()

    except Exception as e:
        print(f"No transcript: {e}")
        return []


# -----------------------------
# 6. TEXT ANALYSIS
# -----------------------------
def text_stats(transcript):
    if not transcript:
        return {"word_count": 0, "unique_words": 0, "top_words": []}

    # FIX HERE 👇 Changed from t['text'] to t.text
    text = " ".join([t.text for t in transcript])

    words = re.findall(r'\w+', text.lower())

    if not words:
        return {"word_count": 0, "unique_words": 0, "top_words": []}

    freq = Counter(words)

    return {
        "word_count": len(words),
        "unique_words": len(set(words)),
        "top_words": freq.most_common(10)
    }


# -----------------------------
# 7. MAIN ANALYSIS
# -----------------------------
def analyze_video(url, prefix):
    print(f"\nProcessing: {url}")

    video_id, title, video_file = download_video(url, prefix)

    if video_file is None or not os.path.exists(video_file):
        raise Exception("Video download failed.")

    # Visual
    frames = extract_frames(video_file)
    brightness = average_brightness(frames)
    cuts = count_cuts(video_file)
    colorfulness = average_colorfulness(frames)
    motion = motion_intensity(video_file)

    # Audio
    audio_file = f"{prefix}.wav"
    extract_audio(video_file, audio_file)
    energy = audio_energy(audio_file)

    # Text
    transcript = get_transcript(video_id)
    text_data = text_stats(transcript)

    return {
        "title": title,
        "brightness": brightness,
        "cuts": cuts,
        "colorfulness": colorfulness,
        "motion_intensity": motion,
        "audio_energy": energy,
        "word_count": text_data["word_count"],
        "unique_words": text_data["unique_words"],
        "top_words": text_data["top_words"]
    }


# -----------------------------
# 8. INSIGHTS
# -----------------------------
def generate_insights(first, latest, rules):
    insights = []
    for rule in rules:
        metric = rule["metric"]
        comparison = rule["comparison"]
        message = rule["message"]
        insight_type = rule.get("type", "neutral")

        if comparison == "increase" and latest[metric] > first[metric]:
            insights.append({"message": message, "type": insight_type})
        elif comparison == "decrease" and latest[metric] < first[metric]:
            insights.append({"message": message, "type": insight_type})
        elif comparison == "change" and latest[metric] != first[metric]:
            insights.append({"message": message, "type": insight_type})
    return insights

# Define default insight rules
default_insight_rules = [
    {"metric": "cuts", "comparison": "increase", "message": "Faster editing in newer video.", "type": "neutral"},
    {"metric": "cuts", "comparison": "decrease", "message": "Slower editing in newer video.", "type": "neutral"},
    {"metric": "audio_energy", "comparison": "increase", "message": "Higher audio energy (more engaging).", "type": "positive"},
    {"metric": "audio_energy", "comparison": "decrease", "message": "Lower audio energy (less intense).", "type": "negative"},
    {"metric": "brightness", "comparison": "increase", "message": "Improved lighting/visual quality.", "type": "positive"},
    {"metric": "brightness", "comparison": "decrease", "message": "Lower average brightness (possibly moodier or darker).", "type": "negative"},
    {"metric": "colorfulness", "comparison": "increase", "message": "More vibrant colors in newer video.", "type": "positive"},
    {"metric": "colorfulness", "comparison": "decrease", "message": "Less vibrant colors in newer video.", "type": "negative"},
    {"metric": "motion_intensity", "comparison": "increase", "message": "Higher motion intensity in newer video (more dynamic visuals).", "type": "positive"},
    {"metric": "motion_intensity", "comparison": "decrease", "message": "Lower motion intensity in newer video (more static visuals).", "type": "negative"},
    {"metric": "word_count", "comparison": "increase", "message": "More content delivered.", "type": "positive"},
    {"metric": "word_count", "comparison": "decrease", "message": "Less content delivered.", "type": "negative"},
    {"metric": "unique_words", "comparison": "increase", "message": "Richer vocabulary.", "type": "positive"},
    {"metric": "unique_words", "comparison": "decrease", "message": "Simpler vocabulary.", "type": "negative"},
    {"metric": "top_words", "comparison": "change", "message": "Changes in top keywords/themes.", "type": "neutral"}
]


# -----------------------------
# 9. VIDEO SUMMARIZATION
# -----------------------------
def summarize_video(video_data):
    summary_parts = []
    summary_parts.append(f"The video '{video_data['title']}' has the following characteristics:")

    summary_parts.append(f"- Average brightness: {video_data['brightness']:.2f} ( {'bright' if video_data['brightness'] > 150 else 'dark' if video_data['brightness'] < 100 else 'moderate'} )")
    summary_parts.append(f"- Number of cuts: {video_data['cuts']} ( {'fast-paced' if video_data['cuts'] > 100 else 'slower-paced'} editing )")
    summary_parts.append(f"- Colorfulness: {video_data['colorfulness']:.2f} ( {'vibrant' if video_data['colorfulness'] > 70 else 'subdued'} colors )")
    summary_parts.append(f"- Motion intensity: {video_data['motion_intensity']:.2f} ( {'dynamic' if video_data['motion_intensity'] > 10 else 'static'} visuals )")
    summary_parts.append(f"- Audio energy: {video_data['audio_energy']:.4f} ( {'energetic' if video_data['audio_energy'] > 0.05 else 'calm'} audio )")
    summary_parts.append(f"- Word count: {video_data['word_count']} words ({video_data['unique_words']} unique words)")
    if video_data['top_words']:
        top_words_str = ', '.join([f'\'{word}\' ({count})' for word, count in video_data['top_words'][:5]])
        summary_parts.append(f"- Top themes: {top_words_str}")

    return '\n'.join(summary_parts)


def assess_creator_improvement(insights):
    positive_changes = [i["message"] for i in insights if i["type"] == "positive"]
    negative_changes = [i["message"] for i in insights if i["type"] == "negative"]
    neutral_changes = [i["message"] for i in insights if i["type"] == "neutral"]

    assessment = []
    assessment.append("\n=== Creator Improvement Assessment ===")

    if not insights:
        assessment.append("No significant changes observed between the videos.")
    else:
        if len(positive_changes) > len(negative_changes) + 1:
            assessment.append("Overall, the creator appears to have **improved** in several key aspects.")
            if positive_changes:
                assessment.append("Notable improvements include: " + "; ".join(positive_changes) + ".")
            if negative_changes:
                assessment.append("However, some areas show changes that might be considered less optimal: " + "; ".join(negative_changes) + ".")
        elif len(negative_changes) > len(positive_changes) + 1:
            assessment.append("Overall, the creator seems to have made changes that might be considered **less optimal** in some areas.")
            if negative_changes:
                assessment.append("Specific areas of concern: " + "; ".join(negative_changes) + ".")
            if positive_changes:
                assessment.append("Nonetheless, some aspects show positive development: " + "; ".join(positive_changes) + ".")
        else:
            assessment.append("The changes between the videos are a **mix of positive and negative developments**, or mostly neutral.")
            if positive_changes:
                assessment.append("Positive aspects: " + "; ".join(positive_changes) + ".")
            if negative_changes:
                assessment.append("Potential areas for review: " + "; ".join(negative_changes) + ".")
        if neutral_changes:
            assessment.append("Other notable changes: " + "; ".join(neutral_changes) + ".")

    return '\n'.join(assessment)


# -----------------------------
# 10. RUN
# -----------------------------
if __name__ == "__main__":
    first_url = "https://youtu.be/rowKH0AcZ8Q"
    latest_url = "https://youtu.be/dsl6kQYMt0M"

    first = analyze_video(first_url, "first_video")
    latest = analyze_video(latest_url, "latest_video")

    # Exclude 'top_words' for DataFrame display to avoid column containing list issues
    df_data = [
        {k: v for k, v in first.items() if k != 'top_words'},
        {k: v for k, v in latest.items() if k != 'top_words'}
    ]
    df = pd.DataFrame(df_data, index=["First", "Latest"])

    print("\n=== COMPARISON ===")
    print(df)
    print("\n=== TOP WORDS COMPARISON ===")
    print("First video top words:", first["top_words"])
    print("Latest video top words:", latest["top_words"])

    print("\n=== VIDEO SUMMARIES ===")
    print("\n--- First Video Summary ---")
    print(summarize_video(first))
    print("\n--- Latest Video Summary ---")
    print(summarize_video(latest))

    print("\n=== INSIGHTS ===")
    formatted_insights = generate_insights(first, latest, default_insight_rules)
    for i, insight_obj in enumerate(formatted_insights, 1):
        print(f"{i}. {insight_obj['message']}")

    print(assess_creator_improvement(formatted_insights))



Processing: https://youtu.be/rowKH0AcZ8Q
[youtube] Extracting URL: https://youtu.be/rowKH0AcZ8Q
[youtube] rowKH0AcZ8Q: Downloading webpage


[youtube] rowKH0AcZ8Q: Downloading android vr player API JSON
[info] rowKH0AcZ8Q: Downloading 1 format(s): 18
[download] first_video.mp4 has already been downloaded
[download] 100% of   10.80MiB

Processing: https://youtu.be/dsl6kQYMt0M
[youtube] Extracting URL: https://youtu.be/dsl6kQYMt0M
[youtube] dsl6kQYMt0M: Downloading webpage


[youtube] dsl6kQYMt0M: Downloading android vr player API JSON
[info] dsl6kQYMt0M: Downloading 1 format(s): 18
[download] latest_video.mp4 has already been downloaded
[download] 100% of   26.66MiB

=== COMPARISON ===
                                                    title  brightness  cuts  \
First   PUBLIC TRANSPORTATION | ANIMATION VIDEO | ASSA...  194.031794   188   
Latest               First Love Part 2 | Indian Animation  131.294096   180   

        colorfulness  motion_intensity  audio_energy  word_count  unique_words  
First      72.874391         11.675497      0.044344         329           193  
Latest     68.177890         10.969969      0.072345         197           122  

=== TOP WORDS COMPARISON ===
First video top words: [('hein', 13), ('bhaiya', 11), ('nahi', 8), ('khali', 8), ('ke', 7), ('mujhe', 6), ('bhi', 6), ('gari', 6), ('jana', 5), ('to', 5)]
Latest video top words: [('i', 7), ('the', 7), ('to', 7), ('it', 6), ('a', 5), ('video', 5), ('you', 5), ('so', 4), ('

In [ ]:
import re
from collections import Counter
from youtube_transcript_api import YouTubeTranscriptApi

# -----------------------------
# 1. EXTRACT VIDEO ID
# -----------------------------
def extract_video_id(url):
    if "youtu.be/" in url:
        return url.split("/")[-1]
    elif "watch?v=" in url:
        return url.split("v=")[-1].split("&")[0]
    else:
        return url


# -----------------------------
# 2. GET TRANSCRIPT
# -----------------------------
def get_transcript(video_id):
    try:
        api = YouTubeTranscriptApi()

        transcript_list = api.list(video_id)

        # Prefer English transcript
        transcript = transcript_list.find_transcript(['en'])

        data = transcript.fetch()

        return data

    except Exception as e:
        print(f"Transcript error: {e}")
        return None


# -----------------------------
# 3. PROCESS TEXT
# -----------------------------
def process_transcript(transcript):
    text = " ".join([t['text'] for t in transcript])
    words = re.findall(r'\w+', text.lower())

    duration = transcript[-1]['start'] + transcript[-1]['duration']

    return text, words, duration


# -----------------------------
# 4. ANALYSIS
# -----------------------------
def analyze_text(words, duration):
    word_count = len(words)
    unique_words = len(set(words))
    words_per_min = word_count / (duration / 60) if duration > 0 else 0

    freq = Counter(words)
    top_words = freq.most_common(10)

    return {
        "word_count": word_count,
        "unique_words": unique_words,
        "words_per_minute": round(words_per_min, 2),
        "top_words": top_words
    }


# -----------------------------
# 5. INSIGHTS
# -----------------------------
def generate_insights(stats):
    insights = []

    if stats["words_per_minute"] > 160:
        insights.append("Fast-paced speaking (high energy / engagement).")
    elif stats["words_per_minute"] < 100:
        insights.append("Slow-paced speaking (calm or less engaging).")

    if stats["unique_words"] / stats["word_count"] > 0.5:
        insights.append("High vocabulary variety (rich content).")
    else:
        insights.append("Repetitive vocabulary (simpler messaging).")

    return insights


# -----------------------------
# 6. MAIN FUNCTION
# -----------------------------
def run_transcript_analysis(url):
    print(f"\nAnalyzing: {url}")

    video_id = extract_video_id(url)
    transcript = get_transcript(video_id)

    if transcript is None:
        print("No transcript available.")
        return

    text, words, duration = process_transcript(transcript)
    stats = analyze_text(words, duration)

    print("\n=== TRANSCRIPT STATS ===")
    print(f"Word Count: {stats['word_count']}")
    print(f"Unique Words: {stats['unique_words']}")
    print(f"Words per Minute: {stats['words_per_minute']}")

    print("\nTop Words:")
    for word, count in stats["top_words"]:
        print(f"{word}: {count}")

    print("\n=== INSIGHTS ===")
    insights = generate_insights(stats)
    for i, insight in enumerate(insights, 1):
        print(f"{i}. {insight}")


# -----------------------------
# 7. RUN
# -----------------------------
if __name__ == "__main__":
    url = "https://youtu.be/62wEk02YKs0"
    run_transcript_analysis(url)


Analyzing: https://youtu.be/62wEk02YKs0
Transcript error: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=62wEk02YKs0! This is most likely caused by:

No transcripts were found for any of the requested language codes: ['en']

For this video (62wEk02YKs0) transcripts are available in the following languages:

(MANUALLY CREATED)
 - en-GB ("English (United Kingdom)")

(GENERATED)
None

(TRANSLATION LANGUAGES)
None

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!
No transcript available.
